https://ai.pydantic.dev/common_tools/


In [1]:
!pip install -qq 'pydantic-ai-slim[duckduckgo]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.9/138.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.7/293.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.4/88.4 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 137.2 MB/s eta 0:00:00


In [2]:
!pip install -qq 'pydantic-ai-slim[tavily]'

In [4]:
from openai import AsyncAzureOpenAI
from google.colab import userdata

from pprint import pprint
import json
import httpx

import nest_asyncio

from dataclasses import dataclass
from IPython.display import Markdown, display, clear_output

import numpy as np

from typing import Literal, Union

from rich.prompt import Prompt
from rich import print
from pydantic_ai import Agent, Tool
from pydantic_ai.models.openai import OpenAIModel
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from pydantic_ai.messages import ModelMessage

In [5]:
nest_asyncio.apply()
from pydantic_ai.providers.azure import AzureProvider

model = OpenAIModel(userdata.get('AZURE_MODEL_NAME'),
    provider=AzureProvider(
        azure_endpoint=userdata.get('AZURE_BASE_URL'),
        api_version=userdata.get('AZURE_API_VERSION'),
        api_key=userdata.get('AZURE_API_KEY'),
    ),)

In [6]:
from IPython.display import Markdown, display

def printmd(string:str):
    display(Markdown(string))

In [7]:
from pydantic_ai import Agent
from pydantic_ai.common_tools.duckduckgo import duckduckgo_search_tool

agent = Agent(
    model=model,
    tools=[duckduckgo_search_tool(max_results=None)],
    system_prompt='Search DuckDuckGo for the given query and answer it detaily.',
)

result = agent.run_sync(
    'Can you list all winners of OSCARs 2025?'
)
printmd(result.output)

The 97th Academy Awards, held in 2025, saw several notable wins across major categories. Here is the list of the winners:

1. **Best Picture**: "Anora"
2. **Best Director**: Sean Baker for "Anora"
3. **Best Actor in a Leading Role**: Adrien Brody for "The Brutalist"
4. **Best Actress in a Leading Role**: Mikey Madison for "Anora"
5. **Best Actor in a Supporting Role**: Kieran Culkin for "A Real Pain"
6. **Best Actress in a Supporting Role**: Zoe Saldaña
7. **Best Animated Feature Film**: "Flow"
8. **Best Documentary Feature Film**: "No Other Land"
9. Sean Baker also won multiple Oscars for his work on "Anora," including categories for editing and original screenplay.

The awards highlighted standout performances and cinematic achievements, with "Anora" emerging as a major winner, claiming five awards altogether.

In [8]:
from google.colab import userdata
from pydantic_ai.agent import Agent
from pydantic_ai.common_tools.tavily import tavily_search_tool

agent = Agent(
    model=model,
    tools=[tavily_search_tool(userdata.get('TAVILY_API_KEY'))],
    system_prompt='Search Tavily for the given query and return the results.',
)

result = agent.run_sync(
    'Tell me the top news in the GenAI world, give me links.'
    )
printmd(result.output)

Here are some of the top news articles in the Generative AI world:

1. **Generative AI: The New Frontier in Parkinson's and Space Medicine - OpenTools**
   - Read more: [OpenTools Article](https://opentools.ai/news/generative-ai-the-new-frontier-in-parkinsons-and-space-medicine)

2. **NIST’s attempts to secure AI yield many questions, no answers - csoonline.com**
   - Read more: [CSO Online Article](https://www.csoonline.com/article/4042627/nists-attempts-to-secure-ai-yields-many-questions-no-answers.html)

3. **Foreign disinformation enters its AI era — just as U.S. pulls back resources to fight it - Axios**
   - Read more: [Axios Article](https://www.axios.com/2025/08/22/ai-disinformation-china-golaxy-vanderbilt)

4. **Epic unveils AI solutions for clinicians, patients and RCM, gives sneak peek at Cosmos AI for risk prediction - Fierce Healthcare**
   - Read more: [Fierce Healthcare Article](https://www.fiercehealthcare.com/health-tech/epic-unveils-major-ai-features-ai-charting-microsoft-cosmos-ai-risk-prediction-and-rcm)

5. **Quick Hits in AI News: In-Person Interviews Make a Comeback - SHRM**
   - Read more: [SHRM Article](https://www.shrm.org/topics-tools/flagships/ai-hi/quick-hits-august-25)

These articles cover a wide range of topics related to the latest advancements and challenges in Generative AI.

In [10]:
from pydantic import BaseModel
from pydantic_ai import ModelRetry, RunContext
from typing import Dict, List, Union

class News(BaseModel):
    summary: str
    insights: List[str]

class ErrorData(BaseModel):
    error: str

ModelAgent = Union[News, ErrorData]

async def connect_to_local_database(user_inquery: str) -> str:
    if not isinstance(user_inquery, str) or user_inquery=="":
        raise ModelRetry("The user_inquery must be a string and not empty")
    return user_inquery

agent = Agent(
    model=model,
    tools=[connect_to_local_database, duckduckgo_search_tool(max_results=None)],
    instructions="""
    Use connect_to_local_database tool and then
    Search DuckDuckGo for the given query and answer it detaily.
    """,
    output_type=ModelAgent,
)

@agent.output_validator
async def validate_Agent(deps: RunContext[None], output: ModelAgent) -> ModelAgent:
    if isinstance(output, ErrorData):
        raise ModelRetry("You must check again, you should return News object")
    elif isinstance(output, News):
        if len(output.insights) < 10:
            raise ModelRetry("The insights of News object must be more than 10 itema")
        return output
    else:
        raise ModelRetry("You must check again, you should return News object or ErrorData object")

result = agent.run_sync(
    'Find news regarding A2A agents communications'
)
print(result.output.model_dump())

{
    'summary': 'The Agent2Agent (A2A) protocol is revolutionizing AI agent communication by providing a universal 
open standard for inter-agent communication, as initiated by Google in 2025. This has been furthered by 
collaborations with over 100 technology companies, including major players like Microsoft, AWS, and the Linux 
Foundation.',
    'insights': [
        'The A2A protocol enables AI agents to securely exchange information and coordinate actions across various 
enterprise platforms.',
        'Google launched the A2A protocol as a groundbreaking open standard, allowing AI agents to communicate 
seamlessly with each other.',
        'This protocol is supported and enhanced by major technology companies such as Microsoft and AWS.',
        'The Linux Foundation has also adopted the A2A protocol, expanding its influence in multi-agent systems.',
        'A2A aims to break down silos between different ecosystems, allowing specialized agents to collaborate on 
tasks that a single agent cannot handle alone.',
        'Microsoft integrated the A2A protocol with tools like Semantic Kernel and LangChain, enhancing structured 
agent communication.',
        'Everest Group highlights the A2A protocol as part of a strong ecosystem with over 100 partners.',
        "The protocol supports stateless interactions, simplifying development where session management isn't 
needed.",
        "With Google's ADK open source framework, building A2A agents has become more accessible.",
        "The A2A protocol's collaborative nature is seen as de facto standard for AI agents on the web.",
        'The adoption of A2A by AWS and its inclusion in the Strands Agents SDK marks a significant endorsement of 
the protocol.',
        "The protocol's ability to work across different AI frameworks provides standardized ways for agent 
discovery and negotiation of communication formats.",
        'These advances are reshaping how AI systems coordinate, learn, and expand.',
        'The protocol is seen as pivotal for future AI inter-agent workflows and collaboration.',
        'A2A enables not only agent-to-agent communication but also human-to-agent interactions within the same 
protocol, which expands its utility.',
        'Leading tech companies view A2A as essential for navigating complex multi-agent environments.',
        'Protocols like A2A are critical for scaling AI communications in dynamic enterprise scenarios.',
        "The ongoing development and updates by Google continue to enhance A2A's capabilities, including security 
upgrades."
    ]
}

In [11]:
print(len(result.output.insights))

18

In [12]:
print(result.all_messages())

[
    ModelRequest(parts=[UserPromptPart(content='Find news regarding A2A agents communications', 
timestamp=datetime.datetime(2025, 8, 25, 15, 55, 51, 411965, tzinfo=datetime.timezone.utc))], instructions='Use 
connect_to_local_database tool and then\n    Search DuckDuckGo for the given query and answer it detaily.'),
    ModelResponse(parts=[ToolCallPart(tool_name='connect_to_local_database', args='{"user_inquery": "A2A agents 
communications news"}', tool_call_id='call_Ne1foEfNWSNNTSTzqM9pcfkw'), ToolCallPart(tool_name='duckduckgo_search', 
args='{"query": "A2A agents communications news"}', tool_call_id='call_UzUCgaF50KHDM5FRyKjvQGi1')], 
usage=RequestUsage(input_tokens=167, output_tokens=62, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0,
'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}), model_name='gpt-4o-2024-08-06', 
timestamp=datetime.datetime(2025, 8, 25, 15, 55, 51, tzinfo=TzInfo(UTC)), provider_name='azure', 
provider_request_id='chatcmpl-C8TpvEh6hEyg2LOqB4h6tujTbvDay'),
    ModelRequest(parts=[ToolReturnPart(tool_name='connect_to_local_database', content='A2A agents communications 
news', tool_call_id='call_Ne1foEfNWSNNTSTzqM9pcfkw', timestamp=datetime.datetime(2025, 8, 25, 15, 55, 52, 325740, 
tzinfo=datetime.timezone.utc)), ToolReturnPart(tool_name='duckduckgo_search', content=[{'title': 'Google Developers
Announcing the Agent2Agent Protocol (A2A) - Google Developers Blog', 'href': 
'https://developers.googleblog.com/en/a2a-a-new-era-of-agent-interoperability/', 'body': 'April 9, 2025 - Today, 
we’re launching a new, ... PwC, TCS, and Wipro. The A2A protocol will allow AI agents to communicate with each 
other, securely exchange information, and coordinate actions on top of various enterprise platforms or applications
....'}, {'title': 'Microsoft Empowering multi-agent apps with the open Agent2Agent (A2A) protocol | The Microsoft 
Cloud Blog', 'href': 
'https://www.microsoft.com/en-us/microsoft-cloud/blog/2025/05/07/empowering-multi-agent-apps-with-the-open-agent2ag
ent-a2a-protocol/', 'body': 'May 22, 2025 - A2A can enable structured agent communication—exchanging goals, 
managing state, invoking actions, and returning results securely and observably . Developers can use tools they 
know, like Semantic Kernel or LangChain, and still interoperate. Every call travels through enterprise-grade 
safeguards: ...'}, {'title': 'Linux Foundation Linux Foundation Launches the Agent2Agent Protocol Project to Enable
Secure, Intelligent Communication Between AI Agents', 'href': 
'https://www.linuxfoundation.org/press/linux-foundation-launches-the-agent2agent-protocol-project-to-enable-secure-
intelligent-communication-between-ai-agents', 'body': 'June 23, 2025 - The A2A protocol is a collaborative effort 
launched by Google in April and with growing support from more than 100 leading technology companies . The protocol
addresses the growing need for agents to operate in dynamic, multi-agent environments, coordinating actions across 
a wide array of ...'}, {'title': 'GitHub GitHub - a2aproject/A2A: An open protocol enabling communication and 
interoperability between opaque agentic applications.', 'href': 'https://github.com/a2aproject/A2A', 'body': 'A2A 
aims to: Break Down Silos: Connect agents across different ecosystems . Enable Complex Collaboration: Allow 
specialized agents to work together on tasks that a single agent cannot handle alone. Promote Open Standards: 
Foster a community-driven approach to agent communication, encouraging ...'}, {'title': 'A2A Protocol A2A Protocol 
- Agent2Agent Communication', 'href': 'https://a2aprotocol.ai/', 'body': "Agents can send messages to communicate 
context, replies, artifacts, or user instructions . ... Messages include 'parts' with specified content types, 
allowing agents to negotiate the correct format and UI capabilities. The A2A Protocol follows a well-defined flow 
for agent communication and ..."}, {'title': 'Everest Group The Rise Of Agent Protoco